In [6]:
import torch 
import torch.nn as nn 
import tiktoken 

In [7]:
GPT_CONFIG = {
    "vocab_size" : 50527,
    "context_length" : 1024,
    "emb_dim" : 768,
    "n_heads" : 12,
    "n_layers" : 12,
    "drop_rate" : 0.1,
    "qkv_bias" : False, 
}

In [10]:
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")

txt1 = "How are we going to do that???"          
txt2 = "I do not know"   

tok1 = tokenizer.encode(txt1)
tok2 = tokenizer.encode(txt2)

max_len = max(len(tok1), len(tok2))

pad_token_id = tokenizer.eot_token

tok1_padded = tok1 + [pad_token_id] * (max_len - len(tok1))
tok2_padded = tok2 + [pad_token_id] * (max_len - len(tok2))

batch = torch.tensor([tok1_padded, tok2_padded])
print(batch)  

tensor([[ 2437,   389,   356,  1016,   284,   466,   326, 28358],
        [   40,   466,   407,   760, 50256, 50256, 50256, 50256]])


In [12]:
torch.manual_seed(123)

batch_example = torch.rand(2 , 5)

batch_example

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740],
        [0.8665, 0.1366, 0.1025, 0.1841, 0.7264]])

In [14]:
layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())  
out = layer(batch_example)
out

tensor([[0.0000, 0.0000, 0.4091, 0.6587, 0.3914, 0.0000],
        [0.0000, 0.0000, 0.1902, 0.3182, 0.6486, 0.0000]],
       grad_fn=<ReluBackward0>)

In [19]:
mean = out.mean(dim = 1, keepdim = True)
mean

tensor([[0.2432],
        [0.1928]], grad_fn=<MeanBackward1>)

In [20]:
var = out.var(dim = 1, keepdim = True)
var

tensor([[0.0799],
        [0.0670]], grad_fn=<VarBackward0>)

In [23]:
normed = (out - mean ) / torch.sqrt(var)
normed.var(dim = 1, keepdim = True)

tensor([[1.],
        [1.]], grad_fn=<VarBackward0>)

In [24]:
torch.set_printoptions(sci_mode = False)

In [25]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim = 1, keepdim = True)
        var = x.var(dim = 1, keepdim = True, unbiased = False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x * self.shift

In [ ]:
import torch
import torch.nn as nn


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x


class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim = 1, keepdim = True)
        var = x.var(dim = 1, keepdim = True, unbiased = False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x * self.shift

In [ ]:
torch.manual_seed(123)

model = GPTModel(cfg = GPT_CONFIG)

logits = model(batch)

print("output shape", logits.shape)
print(logits) 